In [ ]:
# import the necessary python modules
import numpy as np
import time
import corner
from astropy.cosmology import FlatLambdaCDM

import lenstronomy

from lenstronomy.LensModel.lens_model import LensModel
from lenstronomy.LensModel.lens_model_extensions import LensModelExtensions
from lenstronomy.LensModel.Solver.lens_equation_solver import LensEquationSolver
from lenstronomy.Cosmo.lens_cosmo import LensCosmo
from lenstronomy.Util import constants
from lenstronomy.Util import param_util
from lenstronomy.Plots import lens_plot

import matplotlib.pyplot as plt
%matplotlib inline

import pickle

Notebook inspired by: https://github.com/lenstronomy/lenstronomy-tutorials/blob/main/Notebooks/LensModeling/modelling_of_catalogue_data.ipynb

## Redirect MCMC output to .txt file to not clutter the notebook later on

In [ ]:
import sys
import os
import contextlib

@contextlib.contextmanager
def suppress_stdout_stderr(to_file):
    """
    Redirect stdout and stderr to a file.
    """
    with open(to_file, 'w') as f:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        sys.stdout = f
        sys.stderr = f
        try:
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

def get_median_and_uncertainties(samples):
    median = np.percentile(samples, 50)
    lower = median - np.percentile(samples, 16)
    upper = np.percentile(samples, 84) - median
    return median, lower, upper

def extract_profile(samples, param_names, param_mcmc):
    """Return lens center chains for one profile."""
    idx = [param_mcmc.index(p) for p in param_names]
    x_chain = samples[:, idx[0]]
    y_chain = samples[:, idx[1]]
    return x_chain, y_chain

In [ ]:
# Load model data

class CustomUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # Handle missing LikelihoodAddition class
        if name == 'LikelihoodAddition':
            class LikelihoodAddition:
                pass
            return LikelihoodAddition
        return super().find_class(module, name)

filename = f"joint_modeling/{system_name}/{system_name}_joint.pkl"

with open(filename, "rb") as f:
    loaded_data = CustomUnpickler(f).load()

kwargs_result = loaded_data["kwargs_result"]
chain_list = loaded_data["chain_list"]
sampler_type, samples, param_mcmc, dist_mcmc = chain_list[-1]

param_names = [
                "center_x_lens_light0",
                "center_y_lens_light0"
                ]
# get standard deviation of center from full image modeling
x_lens_chain, y_lens_chain = extract_profile(samples, param_names, param_mcmc)

x_lens, x_lens_upper, x_lens_lower = get_median_and_uncertainties(x_lens_chain)
y_lens, y_lens_upper, y_lens_lower = get_median_and_uncertainties(y_lens_chain)

x_sigma = 0.5 * (x_lens_upper + x_lens_lower)
y_sigma = 0.5 * (y_lens_upper + y_lens_lower)

print(f'1-sigma uncertainties for x and y lens: {x_sigma}, {y_sigma}')

if include_perturber == True:
    ra_deflector = kwargs_result['kwargs_lens'][1]['center_x'] - kwargs_result['kwargs_lens'][0]['center_x']
    dec_deflector = kwargs_result['kwargs_lens'][1]['center_y'] - kwargs_result['kwargs_lens'][0]['center_y']

    param_names = [
                "center_x_lens_light1",
                "center_y_lens_light1"
                ]
    x_def_chain, y_def_chain = extract_profile(samples, param_names, param_mcmc)

    x_def, x_def_upper, x_def_lower = get_median_and_uncertainties(x_def_chain)
    y_def, y_def_upper, y_def_lower = get_median_and_uncertainties(y_def_chain)
    
    x_def_sigma = 0.5 * (x_def_upper + x_def_lower)
    y_def_sigma = 0.5 * (y_def_upper + y_def_lower)


    print(f'1-sigma uncertainties for x and y perturber: {x_def_sigma}, {y_def_sigma}')

# image position in coordinates where lens is centered at (0, 0)
ra_im1 = kwargs_result['kwargs_ps'][0]['ra_image'][0] - kwargs_result['kwargs_lens'][0]['center_x']
dec_im1 = kwargs_result['kwargs_ps'][0]['dec_image'][0] - kwargs_result['kwargs_lens'][0]['center_y']

ra_im2 = kwargs_result['kwargs_ps'][0]['ra_image'][1] - kwargs_result['kwargs_lens'][0]['center_x']
dec_im2 = kwargs_result['kwargs_ps'][0]['dec_image'][1] - kwargs_result['kwargs_lens'][0]['center_y']

image_sep = np.sqrt((ra_im1 - ra_im2)**2 + (dec_im1 - dec_im2)**2)

ximg = np.array([ra_im1, ra_im2])
yimg = np.array([dec_im1, dec_im2])

In [ ]:
# lens model choices
lens_model_list = ['EPL', 'SHEAR']
if include_perturber == True:
    lens_model_list = ['EPL', 'EPL', 'SHEAR']

fixed_lens = []
kwargs_lens_init = []
kwargs_lens_sigma = []
kwargs_lower_lens = []
kwargs_upper_lens = []

# primary lens
kwargs_lens_init.append({
    'theta_E': image_sep / 2,
    'gamma': 2.0,
    'center_x': 0.0,
    'center_y': 0.0,
    'e1': 0.0,
    'e2': 0.0,
})

kwargs_lens_sigma.append({
    'theta_E': 0.1,
    'e1': 0.1,
    'e2': 0.1,
    'center_x': 0.004,
    'center_y': 0.004,
})

kwargs_lower_lens.append({
    'theta_E': 0.1,
    'e1': -0.4,
    'e2': -0.4,
    'center_x': -5,
    'center_y': -5,
})

kwargs_upper_lens.append({
    'theta_E': 4.0,
    'e1': 0.4,
    'e2': 0.4,
    'center_x': 5,
    'center_y': 5,
})

fixed_lens.append({'gamma': 2.0})

# add perturber
if include_perturber == True:
    kwargs_lens_init.append({
        'theta_E': image_sep / 4,
        'gamma': 2.0,
        'center_x': ra_deflector,
        'center_y': dec_deflector,
        'e1': 0.0,
        'e2': 0.0,
    })

    kwargs_lens_sigma.append({
        'theta_E': 0.1,
        'e1': 0.1,
        'e2': 0.1,
        'center_x': 0.004,
        'center_y': 0.004
    })

    kwargs_lower_lens.append({
        'theta_E': 0.05,
        'e1': -0.4,
        'e2': -0.4,
        'center_x': -5,
        'center_y': -5
    })

    kwargs_upper_lens.append({
        'theta_E': 2.0,
        'e1': 0.4,
        'e2': 0.4,
        'center_x': 5,
        'center_y': 5
    })

    fixed_lens.append({
        'gamma': 2.0
    })

# shear parameters
kwargs_lens_init.append({'gamma1': 0.0, 'gamma2': 0.0})
kwargs_lens_sigma.append({'gamma1': 0.05, 'gamma2': 0.05})

if increase_shear == True:
    kwargs_lower_lens.append({'gamma1': -0.25, 'gamma2': -0.25})
    kwargs_upper_lens.append({'gamma1': 0.25, 'gamma2': 0.25})
else:
    kwargs_lower_lens.append({'gamma1': -0.106, 'gamma2': -0.106})
    kwargs_upper_lens.append({'gamma1': 0.106, 'gamma2': 0.106})

fixed_lens.append({'ra_0': 0.0, 'dec_0': 0.0})

lens_params = [kwargs_lens_init, kwargs_lens_sigma, fixed_lens,
               kwargs_lower_lens, kwargs_upper_lens]

# image position parameters
point_source_list = ['LENSED_POSITION']

kwargs_ps_init = [{'ra_image': ximg, 'dec_image': yimg}]
fixed_ps = [{}]
kwargs_ps_sigma = [{'ra_image': [0.001] * 2, 'dec_image': [0.001] * 2}]
kwargs_lower_ps = [{'ra_image': -0.004 + ximg, 'dec_image': -0.004 + yimg}]
kwargs_upper_ps = [{'ra_image': 0.004 + ximg, 'dec_image': 0.004 + yimg}]

# combine all parameter options for lenstronomy
ps_params = [kwargs_ps_init, kwargs_ps_sigma, fixed_ps, kwargs_lower_ps, kwargs_upper_ps]

# Model choices
kwargs_model = {
    'lens_model_list': lens_model_list,
    'point_source_model_list': point_source_list
}

# Imaging data
kwargs_data_joint = {'ra_image_list': [ximg], 'dec_image_list': [yimg]} # assume inputs from Full Image Modeling are the truth

# Likelihood
kwargs_likelihood = {
    'check_bounds': True,
    'source_position_likelihood': True, # evaluates how close the different image positions match the source positons
    'source_position_tolerance': 0.001,
    'astrometric_likelihood': True, # evaluates the astrometric uncertainty of the predicted and modeled image positions with an offset ‘delta_x_image’ and ‘delta_y_image’
    'image_position_uncertainty': 0.004, # uncertainty in image position
    'image_position_likelihood': True # assume inputs from Full Image Modeling are the truth
}

# create centroid distribution that follows full image modeling
if include_perturber == True:
    kwargs_likelihood['prior_lens'] = [
        [0, 'center_x', 0, x_sigma],
        [0, 'center_y', 0, y_sigma],
        [1, 'center_x', ra_deflector, x_def_sigma],
        [1, 'center_y', dec_deflector, y_def_sigma],
    ]
else:
    kwargs_likelihood['prior_lens'] = [
        [0, 'center_x', 0, x_sigma],
        [0, 'center_y', 0, y_sigma]
    ]

# Constraints
kwargs_constraints = {
    'num_point_source_list': [2],
    'point_source_offset': True # for astrometric uncertainty
}

# Special components - astrometric uncertainty
kwargs_special_init = {
    'delta_x_image': [0.0, 0.0],
    'delta_y_image': [0.0, 0.0],
}

kwargs_special_sigma = {
    'delta_x_image': [0.004, 0.004],
    'delta_y_image': [0.004, 0.004],
}

kwargs_special_lower = {
    'delta_x_image': [-0.004, -0.004],
    'delta_y_image': [-0.004, -0.004],
}

kwargs_special_upper = {
    'delta_x_image': [0.004, 0.004],
    'delta_y_image': [0.004, 0.004],
}

kwargs_special_fixed = {}

kwargs_special = [
    kwargs_special_init, 
    kwargs_special_sigma, 
    kwargs_special_fixed, 
    kwargs_special_lower, 
    kwargs_special_upper
]


# combined params
kwargs_params = {
    'lens_model': lens_params,
    'point_source_model': ps_params,
    'special': kwargs_special
}


In [ ]:
from lenstronomy.Workflow.fitting_sequence import FittingSequence
fitting_seq = FittingSequence(kwargs_data_joint, kwargs_model, kwargs_constraints, kwargs_likelihood, kwargs_params, verbose = False)

fitting_kwargs_list = [
                       ['PSO', {'sigma_scale': 1., 'n_particles': 200, 'n_iterations': 500}]
                    ]

start_time = time.time()
chain_list_pso = fitting_seq.fit_sequence(fitting_kwargs_list)
kwargs_result = fitting_seq.best_fit()
end_time = time.time()
print(end_time - start_time, 'total time needed for computation')
print('============ CONGRATULATION, YOUR JOB WAS SUCCESSFUL ================ ')

In [ ]:
kwargs_result = fitting_seq.best_fit(bijective=True)
args_result = fitting_seq.param_class.kwargs2args(**kwargs_result)
logL = fitting_seq.likelihoodModule.logL(args_result, verbose=True)

from lenstronomy.Plots import chain_plot
for i in range(len(chain_list_pso)):
    chain_plot.plot_chain_list(chain_list_pso, i)

plt.show()

In [ ]:
#and now we run the MCMC
fitting_kwargs_list = [
    ['MCMC', {'n_burn': 1000, 'n_run': 10000, 'walkerRatio': 10,'sigma_scale': 0.1}]
]

with suppress_stdout_stderr(f"conjugate_point/{system_name}/{system_name}_conjugate_mcmc_chain.txt"):
        chain_list_mcmc = fitting_seq.fit_sequence(fitting_kwargs_list)

kwargs_result = fitting_seq.best_fit()

In [ ]:
chain_plot.plot_chain_list(chain_list_mcmc)
plt.show()

In [ ]:
sampler_type, samples_mcmc, param_mcmc, dist_mcmc  = chain_list_mcmc[0]

print("number of non-linear parameters in the MCMC process: ", len(param_mcmc))
print("parameters in order: ", param_mcmc)
print("number of evaluations in the MCMC process: ", np.shape(samples_mcmc)[0])

# import the parameter handling class #
from lenstronomy.Sampling.parameters import Param
import lenstronomy.Util.param_util as param_util
# make instance of parameter class with given model options, constraints and fixed parameters
# this allows to recover the full parameters of all model components, not just the ones being sampled.

param = Param(kwargs_model, fixed_lens, kwargs_fixed_ps=fixed_ps,
              kwargs_lens_init=kwargs_result['kwargs_lens'], **kwargs_constraints)
# the number of non-linear parameters and their names #
num_param, param_list = param.num_param()


lensModel = LensModel(kwargs_model['lens_model_list'])
lensModelExtensions = LensModelExtensions(lensModel=lensModel) 

mcmc_new_list = []
labels_new = [r"$\theta_E$", r"$\phi_{lens}$", r"$q$", r"$\phi_{ext}$", r"$\gamma_{ext}$", r"$\alpha_1$", r"$\delta_1$", r"$\alpha_2$", r"$\delta_2$"]

    
print(labels_new)

In [ ]:
for i in range(len(samples_mcmc)):
    # transform the parameter position of the MCMC chain in a lenstronomy convention with keyword arguments 
    kwargs_out = param.args2kwargs(samples_mcmc[i])
    kwargs_lens_out = kwargs_out['kwargs_lens']
    kwargs_ps_out = kwargs_out['kwargs_ps']
    
    # extract quantities of the main deflector
    theta_E = kwargs_lens_out[0]['theta_E']
    e1, e2 = kwargs_lens_out[0]['e1'], kwargs_lens_out[0]['e2']
    phi, q = param_util.ellipticity2phi_q(e1, e2)
    if system_name == 'J1001+5027':
        gamma1, gamma2 = kwargs_lens_out[2]['gamma1'], kwargs_lens_out[2]['gamma2']
    else:
        gamma1, gamma2 = kwargs_lens_out[1]['gamma1'], kwargs_lens_out[1]['gamma2']
        
    phi_ext, gamma_ext = param_util.shear_cartesian2polar(gamma1, gamma2)

    # create image position posterior
    #ra_im1 = kwargs_ps_out[0]['ra_image'][0]
    #dec_im1 = kwargs_ps_out[0]['dec_image'][0]

    #ra_im2 = kwargs_ps_out[0]['ra_image'][1]
    #dec_im2 = kwargs_ps_out[0]['dec_image'][1]
    
    new_chain = [theta_E, phi, q, phi_ext, gamma_ext, 
                 #ra_im1, dec_im1, ra_im2, dec_im2
                ]
    
    mcmc_new_list.append(np.array(new_chain))

In [ ]:
plot = corner.corner(np.array(mcmc_new_list), labels=labels_new, show_titles=True)
plt.show()

In [ ]:
output_file = f"conjugate_point/{system_name}/{system_name}_conjugate.pkl"

# package results in a dictionary
results_dict = {
    "kwargs_result": kwargs_result,
    "kwargs_model": kwargs_model,
    "samples_raw": samples_mcmc,
    "kwargs_constraints": kwargs_constraints,
    "chain_list": chain_list_mcmc,
    "processed_chain": np.array(mcmc_new_list),
    "labels": labels_new,
    "param_list": param_mcmc          
}

# save
with open(output_file, "wb") as f:
    pickle.dump(results_dict, f)

print(f"MCMC results saved to {output_file}")